# Task 3

Using the database from Task 1, write Python code that:
- Sends queries and extracts results.
- Reads one of the relations that contains an attribute with strings (e.g. names) and store it in a Pandas DataFrame (df_sim).
- Defines a Hashing Function that hashes the strings into indexes between 0 and m-1, where m is the table cardinality. Report the collision rate of the function on the available data.

In [1]:
#Imports
import sqlite3
from pathlib import Path

import pandas as pd

# Build the database from the three SQL files in ../sql (schema, data, queries)
sql_dir = Path("..") / "sql"
sql_files = ["01_schema.sql", "02_data.sql", "03_queries.sql"]

# New empty SQLite database to work with (ignored by git, rebuilt on every run)
db_file = Path("..") / "tennis_assignment.db"
db_file.unlink(missing_ok=True)

# Read the SQL code of all files
sql_code = "\n".join((sql_dir / name).read_text(encoding="utf-8") for name in sql_files)

sql_code = "\n".join(
    line
    for line in sql_code.splitlines()
    if not line.strip().startswith(".") # does not include the lines that start with a dot, which are command-line program commands
)

#Connect to data base, put all the data in from SQL code
conn = sqlite3.connect(db_file)
conn.executescript(sql_code)


## Part 1
Creates a connection to the database file that was created in Task 1, send SQL queries to the database and extracts the results.

In [2]:
#Cursor to send SQL queries from python to SQL
cursor = conn.cursor()

#Query to test if it's working
#Selects the first 5 players and gives their id and name
query = "SELECT player_id, player_name FROM player LIMIT 5;"

cursor.execute(query) #Sends SQL query back to database

results = cursor.fetchall() #Fetchall brings results back to python

print(results)


[('A0E2', 'Carlos Alcaraz'), ('A0FC', 'Matteo Arnaldi'), ('A0GC', 'Terence Atmane'), ('A0K3', 'Nicolas Arseneault'), ('AE14', 'Daniel Altmaier')]


## Part 2
Read one of the relations that contains an attribute with strings and store it in a Pandas DataFrame (df_sim).

In [3]:
# We're using the table coach.
# This table contains "coach_full_name" which is a string attribute and coach_id.


df_sim = pd.read_sql_query( #does the sql query and puts result in a df
    "SELECT coach_id, coach_full_name FROM coach ORDER BY coach_id;",
    conn
)


#Total amount of coaches in table
print(f"Total amount of coaches in table: {len(df_sim)}")

#Show first 10 rows of df_sim
display(df_sim.head(10))



Total amount of coaches in table: 249


,coach_id,coach_full_name
0,1,Adam Kellner
1,2,Admir Jasika
2,3,Adolfo Gutierrez
3,4,Alberto Mancini
4,5,Alberto Martin
5,6,Alejandro Fabbri
6,7,Alejandro Gonzalez
7,8,Alen Avidzba
8,9,Alessandro Bega
9,10,Alessandro Petrone


## Part 3
Hashing Function that hashes the strings into indexes between 0 and m-1, where m is the table cardinality. Report the collision rate of the function on the available data.

In [4]:
# m = cardinality of the table (# of rows in df_sim or # amount of coaches in our case).
m = len(df_sim)


# Make hash index out of string. Has to be between 0 and m-1.
def hash_string(text, m):

    # Start with hash_value = 0
    hash_value = 0

    # Go through every character
    for character in text:

        #ord gives every character a nr and adds it to hash value.
        hash_value += ord(character)

    # % m guarantees that the result is always within m-1
    return hash_value % m


# Put each coach name (coach_full_name) through hash function
# The results are saved in a new column "hash_index"

def hash(name):
    return hash_string(name, m)

df_sim["hash_index"] = df_sim["coach_full_name"].apply(hash)

# nunique() counts how many different (=unique) indexes there are
# some might have been used multiple times
unique_hashes = df_sim["hash_index"].nunique()


# Calculating collision by doing # of strings - unique strings
collisions = m - unique_hashes


# Calculating and printing collision rate
collision_rate = collisions / m
print(f"Collision rate: {collision_rate:.2%}")


Collision rate: 39.36%


In [5]:
# we can see the similar names we added on purpose on the coach table
similar_coach_names = ["Adam Kellner", "Adam Kelner", "Adam Keller", "Kellner Adam", "Adam Kellnre", "Adan Kellner"]

df_similar_coach_names = df_sim[df_sim["coach_full_name"].isin(similar_coach_names)] # keep only the rows where the coach_full_name is in the list of similar names
display(df_similar_coach_names)

,coach_id,coach_full_name,hash_index
0,1,Adam Kellner,124
244,245,Adam Kelner,16
245,246,Adam Keller,14
246,247,Kellner Adam,124
247,248,Adam Kellnre,124
248,249,Adan Kellner,125


We can see that out of the 6 similar names, 3 share the same hashing index. Our hash function just sums the character codes and ignores the order of the letters so that's why names with same letters but different order get the same index.

In [6]:
conn.close()